In [0]:

from pyspark.sql import functions as f 
import uuid 
from delta.tables import DeltaTable
from datetime import datetime

In [0]:
spark.sql("use catalog project")
spark.sql("create schema if not exists bronze_layer")


In [0]:
spark.sql("""
          create table if not exists bronze_layer.ingestion_Control(
              layer string ,
              tablename string ,
              ts_Col  string , 
              pk_col string , 
              last_Successful_ts timestamp,
              last_Successful_pk bigint,
              last_run_id  string , 
              rows_written bigint,
              run_Status string,
              updated_At timestamp




          )
          using delta
          
          """)

In [0]:
tables_config = {
    "orders": {
        "pk_Col": "order_id", 
        "ts_Col": "updated_at"
    },
    "payments": {
        "pk_Col": "payment_id", 
        "ts_Col": "processed_at"
    },
    "product": {
        "pk_Col": "product_id", 
        "ts_Col": "updated_at"
    }
}

bronze_run_id = str(uuid.uuid4())

In [0]:
print(bronze_run_id)

In [0]:
def get_latest_successful_watermark(tablename: str):
    ctrl = (spark.table("project.bronze_layer.ingestion_control")
            .filter((f.col("tablename") == tablename) & 
                    (f.col("run_Status") == "Success") & 
                    (f.col("layer") == "bronze"))
            .orderBy(f.col("updated_At").desc())
            .select("last_successful_ts", "last_successful_pk") # تحديد الأعمدة صراحة لتقليل الـ Overhead
            .limit(1))
    
    rows = ctrl.collect()
    
    if not rows:
        return None, None
    else:
        row_dict = rows[0].asDict()
        ts_val = row_dict.get("last_successful_ts")
        pk_val = row_dict.get("last_successful_pk")
        
        final_ts = ts_val if ts_val is not None else None
        final_pk = int(pk_val) if pk_val is not None else None
        
        return final_ts, final_pk

In [0]:
def upsert_bronze_control(tablename, pk_col, ts_col, last_ts, last_pk, last_run_id, rows_written, run_status, updated_At):
    contro_df = spark.createDataFrame([(
        "bronze",
        tablename,
        ts_col,
        pk_col,
        last_ts,
        int(last_pk) if last_pk is not None else None,
        last_run_id,
        int(rows_written) if rows_written is not None else 0,
        run_status,
        updated_At
    )], schema="""
    layer string,
    tablename string,
    ts_Col string, 
    pk_col string, 
    last_Successful_ts timestamp,
    last_Successful_pk bigint,
    last_run_id string, 
    rows_written bigint,
    run_Status string,
    updated_At timestamp
    """)
    
    dt = DeltaTable.forName(spark, "project.bronze_layer.ingestion_control")
    dt.alias("t").merge(
        contro_df.alias("s"),
        "t.tablename = s.tablename and t.layer = s.layer"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
from datetime import timezone

In [0]:
for table_name, conf in tables_config.items():
    pr_key = conf["pk_Col"]
    time_stamp = conf["ts_Col"]
    
    sourcetable = f"/Volumes/source/dbo/tables/{table_name}.csv"
    targettable = f"project.bronze_layer.{table_name}_raw"
    
    last_successful_ts, last_successful_pk = get_latest_successful_watermark(table_name)
    print(f"\n--- Processing Table: {table_name} ---")
    print(f"Current Watermarks -> TS: {last_successful_ts}, PK: {last_successful_pk}")
    
    try:
        source_df = (spark.read.format("csv")
                     .option("header", "true")
                     .option("inferSchema", "true")
                     .load(sourcetable)
                     .withColumn(time_stamp, f.expr(f"try_cast({time_stamp} as timestamp)"))
                     .filter(f.col(time_stamp).isNotNull() & f.col(pr_key).isNotNull()))
                     
    except Exception as e:
        print(f"Error reading source for {table_name}: {e}")
        continue 

    if last_successful_ts is None:
        print("Cold Start: Loading full table...")
        rows_to_load = source_df
    else:
        time_condition = f.col(time_stamp) > f.lit(last_successful_ts)
        
        if last_successful_pk is not None and str(last_successful_pk).strip() != "":
            pk_condition = (f.col(time_stamp) == f.lit(last_successful_ts)) & (f.col(pr_key).cast("long") > f.lit(int(last_successful_pk)))
            rows_to_load = source_df.filter(time_condition | pk_condition)
        else:
            rows_to_load = source_df.filter(time_condition)
            
    rows_count = rows_to_load.count()
    
    if rows_count == 0:
        print(f"No new rows to load for table {table_name}. Updating control table.")
        upsert_bronze_control(
            tablename=table_name,
            pk_col=pr_key,
            ts_col=time_stamp,
            last_ts=last_successful_ts,
            last_pk=last_successful_pk,
            last_run_id=bronze_run_id,
            rows_written=0,
            run_status="Success",
            updated_At=datetime.now(timezone.utc).replace(tzinfo=None)
        )
        continue
        
    max_ts = rows_to_load.agg(f.max(time_stamp)).collect()[0][0]
    max_pk = (
        rows_to_load
        .filter(f.col(time_stamp) == f.lit(max_ts))
        .agg(f.max(pr_key).cast("long"))
        .collect()[0][0]
    )
    
    final_df = (
        rows_to_load
        .withColumn("bronze_ingested_at", f.current_timestamp())
        .withColumn("bronze_run_id", f.lit(bronze_run_id))
        .withColumn("source_table", f.lit(sourcetable))
        .withColumnRenamed(time_stamp, "bronze_time_stamp")
        .withColumnRenamed(pr_key, "bronze_pr_key")
    )
    
    print(f"Writing {rows_count} rows to {targettable}...")
    final_df.write.format("delta").mode("append").saveAsTable(targettable)
    
    upsert_bronze_control(
        tablename=table_name,
        pk_col=pr_key,
        ts_col=time_stamp,
        last_ts=max_ts,
        last_pk=max_pk,
        last_run_id=bronze_run_id,
        rows_written=rows_count,
        run_status="Success",
        updated_At=datetime.now(timezone.utc).replace(tzinfo=None)
    )
    print(f"Successfully processed {table_name}. New Watermarks -> TS: {max_ts}, PK: {max_pk}")

In [0]:
display(spark.sql("select * from project.bronze_layer.product_raw"))